<a href="https://colab.research.google.com/github/yusuf-codes10/ml-colab-project/blob/main/cleaned_up_magic_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from sklearn.neighbors import KNeighborsClassifier

# Load data (you have to load the .data file)

In [2]:
cols = ["fLength","fWidth","fSize","fConc","fConc1","fAsym","fM3Long","fM3Trans","fAlpha","fDist","class"]
df = pd.read_csv('magic04.data', names=cols)
df['class'] = (df['class'] == 'g').astype(int)  # encode: g=1, h=0
df.head()

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,1
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,1
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,1
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,1
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,1


# scale_dataset function

In [3]:
def scale_dataset(dataframe, oversample=False):
    x = dataframe[dataframe.columns[:-1]].values
    y = dataframe[dataframe.columns[-1]].values

    if oversample:
        ros = RandomOverSampler()
        x, y = ros.fit_resample(x, y)

    scaler = StandardScaler()
    x = scaler.fit_transform(x)
    data = np.hstack((x, np.reshape(y, (-1, 1))))
    return data, x, y

#  Split + scale (THE critical cell — must always run from df)

In [4]:
# Always split from the ORIGINAL df — never from train/valid/test
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df = df_shuffled[:int(0.6*len(df_shuffled))]
valid_df = df_shuffled[int(0.6*len(df_shuffled)):int(0.8*len(df_shuffled))]
test_df  = df_shuffled[int(0.8*len(df_shuffled)):]

train, X_train, y_train = scale_dataset(train_df, oversample=True)
valid, X_valid, y_valid = scale_dataset(valid_df, oversample=False)
test,  X_test,  y_test  = scale_dataset(test_df,  oversample=False)

#  KNN

In [5]:
knn_model = KNeighborsClassifier(n_neighbors=1)
knn_model.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=1)

#  Predict & evaluate

In [6]:
y_pred = knn_model.predict(X_test)
print(y_pred)

[1 0 1 ... 1 1 0]


In [9]:
from sklearn.metrics import classification_report

In [17]:
print(classification_report(y_pred, y_test))

              precision    recall  f1-score   support

           0       0.71      0.73      0.72      1308
           1       0.86      0.85      0.85      2496

    accuracy                           0.81      3804
   macro avg       0.78      0.79      0.79      3804
weighted avg       0.81      0.81      0.81      3804

